# 01 — Chargement des données · Sujet 03 Home Credit

**But de ce carnet** : charger les trois fichiers du sujet, vérifier qu'ils sont complets, et en donner
un premier profil. Il ne choisit ni métrique, ni modèle, ni variables : c'est la suite du projet.

| Fichier | Contenu attendu |
|---|---|
| `train.csv` | 246 008 demandes, 122 colonnes dont `TARGET` (1 = défaut) |
| `a_decider.csv` | 61 503 demandes, 121 colonnes, **sans** `TARGET` |
| `HomeCredit_columns_description.csv` | le dictionnaire des colonnes |

**Où mettre les données** : dans le dossier `Sujet_03_Home_Credit/` à la racine du dépôt (il est ignoré
par Git), ou n'importe où en définissant la variable d'environnement `DONNEES_PROJET`. Le dossier se
télécharge depuis le Drive du cours :
https://drive.google.com/drive/folders/17v1qEMO8FohlDPU99fAjEcuy_nl-Nim5

In [ ]:
# --- 1. Imports et réglages ---------------------------------------------------
from pathlib import Path
import os
import pandas as pd

pd.set_option("display.max_columns", 50)

FICHIERS = {
    "train": "train.csv",
    "a_decider": "a_decider.csv",
    "dictionnaire": "HomeCredit_columns_description.csv",
}
SUJET = "Sujet_03_Home_Credit"
CIBLE = "TARGET"
IDENTIFIANT = "SK_ID_CURR"
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"   # ignoré par Git

In [ ]:
# --- 2. Trouver le dossier de données -----------------------------------------
def trouver_donnees():
    """Cherche le dossier qui contient tous les fichiers du sujet."""
    ici = Path.cwd().resolve()
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    pistes += [ici / SUJET, ici, ici / "donnees", ici.parent / SUJET,
               Path("/kaggle/input"), Path("/content/drive/MyDrive") / SUJET, Path("/content")]
    for racine in pistes:
        candidats = [racine]
        if racine.name == "input" and racine.exists():
            candidats += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidats:
            if all((c / f).is_file() for f in FICHIERS.values()):
                return c
    raise FileNotFoundError(
        f"Dossier de données introuvable. Placez {SUJET}/ à la racine du dépôt "
        "ou définissez DONNEES_PROJET. Chemins essayés : " + ", ".join(map(str, pistes))
    )


DOSSIER = trouver_donnees()
print("Dossier de données :", DOSSIER)
for nom in FICHIERS.values():
    print(f"  {nom:<40} {(DOSSIER / nom).stat().st_size / 1e6:8.1f} Mo")

In [ ]:
# --- 3. Charger, avec un cache Parquet construit une seule fois ----------------
# La première lecture du CSV prend quelques secondes ; les suivantes lisent le
# Parquet, beaucoup plus rapide. Supprimez _cache_parquet/ si les CSV changent.
def charger(nom_fichier):
    source = DOSSIER / nom_fichier
    cache = DOSSIER_CACHE / (Path(nom_fichier).stem + ".parquet")
    if cache.is_file() and cache.stat().st_mtime >= source.stat().st_mtime:
        return pd.read_parquet(cache)
    df = pd.read_csv(source)
    DOSSIER_CACHE.mkdir(exist_ok=True)
    df.to_parquet(cache, index=False)
    return df


train = charger(FICHIERS["train"])
a_decider = charger(FICHIERS["a_decider"])
# le dictionnaire Kaggle n'est pas en UTF-8 : il faut préciser l'encodage
dictionnaire = pd.read_csv(DOSSIER / FICHIERS["dictionnaire"], index_col=0, encoding="latin-1")

print("train      :", train.shape)
print("a_decider  :", a_decider.shape)
print("dictionnaire :", dictionnaire.shape)

In [ ]:
# --- 4. Vérifier que rien n'est tronqué ---------------------------------------
assert train.shape == (246_008, 122), f"train.csv incomplet : {train.shape}"
assert a_decider.shape == (61_503, 121), f"a_decider.csv incomplet : {a_decider.shape}"
assert CIBLE in train.columns and CIBLE not in a_decider.columns
assert set(a_decider.columns) == set(train.columns) - {CIBLE}, "colonnes différentes"
assert train[IDENTIFIANT].is_unique and a_decider[IDENTIFIANT].is_unique
assert not set(train[IDENTIFIANT]) & set(a_decider[IDENTIFIANT]), "identifiants communs"
taux_defaut = train[CIBLE].mean()
print(f"OK — données complètes. Taux de défaut : {taux_defaut:.2%} (attendu : 8,07 %)")

## Premier regard

Les trois cellules suivantes décrivent les données **sans rien modifier** : les manquants ne sont pas
comblés, aucune colonne n'est supprimée.

In [ ]:
# --- 5. Aperçu et types --------------------------------------------------------
display(train.head())
print(train.dtypes.value_counts())

In [ ]:
# --- 6. Cible et valeurs manquantes --------------------------------------------
print(train[CIBLE].value_counts().rename({0: "remboursé (0)", 1: "défaut (1)"}))

manquants = train.isna().mean().sort_values(ascending=False)
print(f"\nColonnes avec plus de 30 % de manquants : {(manquants > 0.30).sum()}")
print(f"Taux maximal de manquants : {manquants.max():.1%}")
manquants.head(15).to_frame("part de manquants")

In [ ]:
# --- 7. Dictionnaire : retrouver le sens d'une colonne --------------------------
def decrire(colonne):
    """Renvoie la définition officielle d'une colonne."""
    lignes = dictionnaire[dictionnaire["Row"] == colonne]
    return lignes[["Row", "Description", "Special"]]


for col in ["EXT_SOURCE_2", "DAYS_BIRTH", "AMT_ANNUITY"]:
    display(decrire(col))

## La suite

Les données sont chargées dans trois objets :

- `train` — les demandes dont l'issue est connue ;
- `a_decider` — les demandes à décider (à transformer en `soumission.csv`, colonnes `SK_ID_CURR,TARGET`) ;
- `dictionnaire` — la définition des colonnes, via `decrire("NOM_COLONNE")`.

Rappel de la règle commune : **ne jamais utiliser `SK_ID_CURR` comme variable**, et ne jamais inventer
la cible des lignes à décider.